In [ ]:

#Odtworzenie slajdu 9: "dN = N+ - N-, bin: 1 day, no t0/lag optimization,
#running window 3350 days (~9 years), step 1 day"


import os
import pandas as pd
import numpy as np
from scipy.stats import binom
import matplotlib.pyplot as plt

# konfiguracja
STACJA = "Oulu_1H"

DESKTOP_PATH = os.path.join(os.path.expanduser("~"), "Desktop")
SCIEZKA_CR = os.path.join(DESKTOP_PATH, "oulu_1H_data.csv")

BASE_DIR = os.path.join(DESKTOP_PATH, "kosmosejsmiczne")
os.makedirs(BASE_DIR, exist_ok=True)

FOLDER_DANE = os.path.join(BASE_DIR, "dane")
SCIEZKA_EQ = os.path.join(FOLDER_DANE, "eq_data_orginal_light.csv")

WYJSCIE_WYKRES = os.path.join(BASE_DIR, f"slajd9_dN_PPDF_{STACJA}.png")
WYJSCIE_CSV = os.path.join(BASE_DIR, f"slajd9_dN_PPDF_{STACJA}.csv")

BIN_SIZE_MINUTES = 1440
WINDOW_DAYS = 3350
WINDOWS_COUNT = WINDOW_DAYS
SHIFT_EQ_BINS = 0
MIN_N = 100


def wczytaj_dane_cr_1h(sciezka):
    df = pd.read_csv(sciezka)
    df["date"] = pd.to_datetime(df["datetime"], errors="coerce", utc=True)
    if df["date"].dt.tz is not None:
        df["date"] = df["date"].dt.tz_localize(None)
    df = df.dropna(subset=["date"])
    df = df.set_index("date").sort_index()

    origin = df.index.min()
    seria = df["value"].resample(f"{BIN_SIZE_MINUTES}min", origin=origin).mean()
    return seria, origin


def wczytaj_dane_eq(sciezka, origin):
    df = pd.read_csv(sciezka)
    df["date"] = pd.to_datetime(df["time"], errors="coerce", utc=True)
    if df["date"].dt.tz is not None:
        df["date"] = df["date"].dt.tz_localize(None)
    df = df.dropna(subset=["date"])
    eq_m4 = df[df["mag"] >= 4.0].set_index("date").sort_index()
    return eq_m4["mag"].resample(f"{BIN_SIZE_MINUTES}min", origin=origin).sum()


# deltaN i PPDF
def licz_dN_i_ppdf(eq_series, cr_series, windows_count, shift_bins=0, min_n=100):
    eq_arr = eq_series.values
    cr_arr = cr_series.values
    idx = eq_series.index
    n_bins = len(idx)
    wyniki = []

    for s in range(0, n_bins - windows_count - abs(shift_bins)):
        eq_start = s + shift_bins if shift_bins >= 0 else s
        cr_start = s if shift_bins >= 0 else s - shift_bins

        cr_bins = cr_arr[cr_start: cr_start + windows_count]
        eq_bins = eq_arr[eq_start: eq_start + windows_count]

        if len(cr_bins) < windows_count or len(eq_bins) < windows_count:
            continue

        cr_delta = np.abs(np.diff(cr_bins))
        eq_sum = eq_bins[1:]
        cr_mean_dopasowane = cr_bins[1:]

        maska = (cr_mean_dopasowane > 0) & (cr_delta > 0)
        eq_f = eq_sum[maska]
        cr_f = cr_delta[maska]

        n_total = len(eq_f)
        if n_total < min_n or np.isnan(eq_f).any() or np.isnan(cr_f).any():
            continue

        med_eq = np.median(eq_f)
        med_cr = np.median(cr_f)
        if med_eq == 0 or med_cr == 0:
            continue

        A = (eq_f / med_eq) - 1
        B = (cr_f / med_cr) - 1
        C = np.sign(A * B)

        n_positive = int(np.sum(C == 1))
        n_negative = int(np.sum(C == -1))
        dN = n_positive - n_negative

        p_val = binom.pmf(n_positive, n_total, 0.5)
        log10_p = np.log10(p_val) if p_val > 0 else np.nan

        t0 = idx[s]
        wyniki.append({"date": t0, "dN": dN, "log10_PPDF": log10_p,
                        "n_total": n_total, "n_positive": n_positive})

    return pd.DataFrame(wyniki)


# plot
def rysuj_wykres(wyniki_df):
    fig, ax1 = plt.subplots(figsize=(14, 6))

    ax1.plot(wyniki_df["date"], wyniki_df["dN"], color="purple", linewidth=0.7,
              label="dN = N+ - N-")
    ax1.set_ylabel("dN = N+ - N-", fontsize=12)
    ax1.set_xlabel("t0 (UTC)", fontsize=12)
    ax1.grid(True, alpha=0.3)
    ax1.axhline(0, color="gray", linewidth=0.5, linestyle="--")

    ax2 = ax1.twinx()
    ax2.plot(wyniki_df["date"], wyniki_df["log10_PPDF"], color="green", linewidth=0.7,
              label="log10(PPDF)")
    ax2.set_ylabel(r"$\log_{10}(PPDF)$", fontsize=12)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=9)

    ax1.set_title(
        f"dN i binomialne PPDF w ruchomym oknie {WINDOW_DAYS} dni\n"
        f"({STACJA}, bin={BIN_SIZE_MINUTES} min = 1 dzień, bez optymalizacji t0/lag)",
        fontsize=12,
    )
    fig.tight_layout()
    plt.savefig(WYJSCIE_WYKRES, dpi=300, bbox_inches="tight")
    plt.show()


# pipeline
def main():
    for sciezka, nazwa in [(SCIEZKA_CR, "CR"), (SCIEZKA_EQ, "EQ")]:
        if not os.path.exists(sciezka):
            print(f"[BŁĄD] Brak pliku {nazwa}: {sciezka}")
            return

    cr_binned, origin = wczytaj_dane_cr_1h(SCIEZKA_CR)
    eq_binned = wczytaj_dane_eq(SCIEZKA_EQ, origin=origin)

    data_start = max(eq_binned.index.min(), cr_binned.index.min())
    data_end = min(eq_binned.index.max(), cr_binned.index.max())

    if (data_end - data_start).days < WINDOW_DAYS:
        print(f"[BŁĄD] Wspólny zakres danych jest krótszy niż wymagane okno ({WINDOW_DAYS} dni).")
        return

    wspolny_indeks = pd.date_range(data_start, data_end, freq=f"{BIN_SIZE_MINUTES}min")
    eq_binned = eq_binned.reindex(wspolny_indeks, fill_value=0)
    cr_binned = cr_binned.reindex(wspolny_indeks)

    wyniki_df = licz_dN_i_ppdf(eq_binned, cr_binned, WINDOWS_COUNT, shift_bins=SHIFT_EQ_BINS, min_n=MIN_N)

    if len(wyniki_df) == 0:
        print("[BŁĄD] Brak wyników.")
        return

    wyniki_df.to_csv(WYJSCIE_CSV, index=False)

    idx_min = wyniki_df["log10_PPDF"].idxmin()
    wiersz = wyniki_df.loc[idx_min]
    print(f"Minimum: log10(PPDF)={wiersz['log10_PPDF']:.2f} przy t0={wiersz['date']}")

    rysuj_wykres(wyniki_df)
    print(f"Wykres zapisany: {WYJSCIE_WYKRES}")


if __name__ == "__main__":
    main()